In [2]:
import os
os.chdir('../../../BertTopic/')

In [4]:
import numpy as np
import pandas as pd
import gensim
from gensim.models import word2vec
from tqdm.auto import tqdm
from baseline.utils.data_preprocessor import Preprocessor, tokenize_ltp

In [81]:
test_filepath = './baseline/data/train.csv'
data_filepath = test_filepath
raw_iter = pd.read_csv(data_filepath)
labels, sentences = [], []
for ind, raw in tqdm(enumerate(raw_iter.values), desc='Data Processing'):
    label, s = raw[-1], raw[1]  # 标签和文本
    s = Preprocessor.basic_pipeline(s)
    s = Preprocessor.process_for_segmented(s)
    sentences.append(s)
    labels.append(label)
    
sentence_tokenized = tokenize_ltp(sentences, user_dict_filepath='./data/user_dict.txt', 
                                  filepath='./data/segmented_cache/segmented_train.txt',postfix='ltp')

Data Processing: 0it [00:00, ?it/s]

缓存文件 ./data/segmented_cache/segmented_train_ltp.pt 存在，直接载入缓存文件！


In [18]:
model = gensim.models.Word2Vec(sentences=sentence_tokenized,  # sentences[0] -> list  sentences[0][0] -> str (token)
                               vector_size=300,  # 生成词向量维度
                               window=5,  # 上下文窗口大小
                               sg=1,  # sg=0时，为CBOW模型，sg=1时，为skip-gram模型
                               hs=0,  # 0时为负采样，1时且negative大于0，则为Hierarchical Softmax
                               negative=5, # 负采样时，负采样的个数，建议在[3,10]之间
                               min_count=2,  # 需要计算词向量的最小词频
                               epochs=10,  # 迭代次数
#                                alpha=0.025,  # 初始学习率
#                                min_alpha=5e-5,  # 学习率最小值
                               seed=1234
                              )

In [50]:
# 获取词表
words = model.wv.index_to_key

In [54]:
len(model.wv.index_to_key)
model.wv.index_to_key[1]

'的'

In [64]:
# 根据指定词获取该词的向量
vec = model.wv[words[1]]
vec.shape
len(model.wv[words[1]])

300

In [39]:
# 计算相似度
model.wv.similarity('新冠','疫情')

0.8103461

In [46]:
# 模型保存
model.save('./result/word2vec/word2vec')
# 模型加载
model = gensim.models.Word2Vec.load('./result/word2vec/word2vec')